### Libraries imports

In [4]:
import boto3
import pickle
import re
import os
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(dotenv_path='.env')

True

### Variables definition

In [5]:
# Fetch AWS credentials from the .env file
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")

# Initialize the S3 client with the credentials
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

model = SentenceTransformer('all-MiniLM-L6-v2')

# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
client = OpenAI()

### 1. Preprocessing the Text Files

In [25]:
def download_file_from_s3(s3, bucket_name, file_key, local_path='/tmp/downloaded_document.txt'):
    s3.download_file(bucket_name, file_key, local_path)
    print(f"Downloaded {file_key} from {bucket_name} to {local_path}")
    with open(local_path, 'r') as file:
        return file.read()

In [56]:
bucket_name = 'aws-test-docs-loka'
file_key = 'sagemaker_documentation/ml-creating-your-amazon-sagemaker-resource.md'
local_path = './ml-creating-your-amazon-sagemaker-resource.md'

text_file = download_file_from_s3(s3, bucket_name, file_key, local_path)


Downloaded sagemaker_documentation/ml-creating-your-amazon-sagemaker-resource.md from aws-test-docs-loka to ./ml-creating-your-amazon-sagemaker-resource.md


In [28]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
        i += chunk_size - overlap
    return chunks

In [9]:
chunks = chunk_text(clean_text(text_file))
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 12


### 2. Generate embeddings

In [29]:
def generate_embeddings(chunks):
    embeddings = model.encode(chunks, convert_to_tensor=False)
    return embeddings

In [12]:
embeddings = generate_embeddings(chunks)
print(f"Generated {len(embeddings)} embeddings.")

Generated 12 embeddings.


### 3. Save Chunks and Embeddings to DynamoDB

#### Delete and create the DynamoDB table again

In [ ]:
# Define the variables
table_name = "aws_test_docs_chunks"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x ../create_dynamo_table.sh

# Run the script from one level above the current working directory
!source ../create_dynamo_table.sh {table_name} {region}

In [35]:
# Initialize the DynamoDB resource and table
dynamodb = boto3.resource('dynamodb', region_name='eu-north-1')
table = dynamodb.Table('aws_test_docs_chunks')

In [41]:
# Function to save chunks and embeddings to DynamoDB with 'faiss_index' as a field
def save_chunks_and_embeddings_to_db(file_key, chunks, embeddings, faiss_index):
    """Save each chunk, its corresponding embedding, and FAISS index to DynamoDB."""
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        unique_id = f'{file_key}_{i}'  # Create unique id by concatenating file_key and chunk index
        table.put_item(
            Item={
                'id': unique_id,  # Unique primary key
                'file_key': file_key,  # The original file the chunk came from
                'faiss_index': int(faiss_index),  # FAISS index
                'chunk': chunk,  # Text chunk
                'embedding': pickle.dumps(embedding)  # Serialized embedding
            }
        )
        faiss_index += 1  # Increment FAISS index
        print(f"DB FAISS index: {faiss_index}")
    print(f"Saved chunks and embeddings for {file_key} to DynamoDB")
    return faiss_index  # Return the updated FAISS index


In [ ]:
save_chunks_and_embeddings_to_db(file_key, chunks, embeddings)

In [37]:
# Check how the chunks are stored
response = table.scan()

print(f"Found {response['Count']} items")

all_embeddings = []

# Iterate through the items and print chunks and embeddings
for item in response['Items']:
    # print(f"Item: {item}")
    chunk = item['chunk']
    file_key = item['file_key']
    faiss_index = item['faiss_index']
    # print(f"Chunk: {chunk}")
    # print(f"File key: {file_key}")
    print(f"faiss_index: {faiss_index}")

Found 283 items
faiss_index: 0
faiss_index: 0
faiss_index: 12
faiss_index: 7
faiss_index: 1
faiss_index: 1
faiss_index: 1
faiss_index: 3
faiss_index: 1
faiss_index: 1
faiss_index: 0
faiss_index: 1
faiss_index: 1
faiss_index: 0
faiss_index: 4
faiss_index: 1
faiss_index: 0
faiss_index: 0
faiss_index: 3
faiss_index: 1
faiss_index: 6
faiss_index: 20
faiss_index: 2
faiss_index: 2
faiss_index: 11
faiss_index: 1
faiss_index: 6
faiss_index: 7
faiss_index: 0
faiss_index: 12
faiss_index: 5
faiss_index: 11
faiss_index: 3
faiss_index: 0
faiss_index: 1
faiss_index: 1
faiss_index: 5
faiss_index: 0
faiss_index: 0
faiss_index: 5
faiss_index: 1
faiss_index: 2
faiss_index: 0
faiss_index: 0
faiss_index: 1
faiss_index: 1
faiss_index: 5
faiss_index: 0
faiss_index: 4
faiss_index: 2
faiss_index: 0
faiss_index: 0
faiss_index: 1
faiss_index: 9
faiss_index: 5
faiss_index: 0
faiss_index: 1
faiss_index: 7
faiss_index: 23
faiss_index: 0
faiss_index: 0
faiss_index: 1
faiss_index: 6
faiss_index: 0
faiss_index: 7
fai

### 4. Create and save FAISS index in S3 bucket

In [31]:
def create_faiss_index(embeddings):
    embeddings_np = np.array(embeddings)
    dimension = embeddings_np.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_np)
    return index

def save_faiss_index_to_s3(faiss_index, s3_bucket, s3_key):
    """Save the FAISS index to S3."""
    local_index_path = './faiss_index.index'  # Save in /tmp directory
    faiss.write_index(faiss_index, local_index_path)  # Save to /tmp directory first
    s3.upload_file(local_index_path, s3_bucket, s3_key)
    print(f"FAISS index saved to S3 at {s3_bucket}/{s3_key}.")

In [23]:
faiss_index = create_faiss_index(embeddings)

save_faiss_index_to_s3(faiss_index, bucket_name, 'faiss_index.index')

FAISS index saved to aws-test-docs-loka/faiss_index.index


### 5. Query the FAISS index

In [7]:
# Load FAISS index from S3
def load_faiss_index_from_s3(s3_bucket, s3_key, local_index_file='faiss_index.index'):
    """Download FAISS index from S3 and load it locally."""
    s3.download_file(s3_bucket, s3_key, local_index_file)
    faiss_index = faiss.read_index(local_index_file)
    return faiss_index

In [6]:
def retrieve_chunks_from_db(query_embedding, faiss_index, top_k=5):
    """Retrieve chunks from DynamoDB using the FAISS indices."""
    
    distances, indices = faiss_index.search(query_embedding, top_k)
    print(f"Distances: {distances}")
    print(f"Indices: {indices}")

    results = []
    source_files = []

    for idx in indices[0]:  # Loop through the top-k indices returned by FAISS
        try:
            """Retrieve chunks from DynamoDB using faiss_index."""

            # Convert FAISS index from numpy.int64 to Python int
            idx = int(idx)
            
            response = table.query(
                IndexName='faiss_index_gsi',  # Use the GSI for faiss_index
                KeyConditionExpression=boto3.dynamodb.conditions.Key('faiss_index').eq(idx)
            )
            print(f"Table item: {response}")
            # Check if any items were returned
            if 'Items' in response and response['Items']:
                print(f"Response Item: {response['Items'][0]}")
                chunk = response['Items'][0]['chunk']
                file_key = response['Items'][0]['file_key']  # Retrieve the associated file key
                
                # Append the chunk and file key to the results
                results.append(chunk)
                source_files.append(file_key)
                
                print(f"Chunk: {chunk}")
                print(f"File key: {file_key}")
            else:
                print(f"No item found for FAISS index: {idx}")
        except Exception as e:
            print(f"Error retrieving chunk for index {idx}: {str(e)}")
    
    return results, source_files

In [10]:
query = "Why Should You Use MLOps?"
query_embedding = model.encode([query])

# Load FAISS index from S3
s3_index_bucket = 'aws-docs-loka-bk'
s3_index_key = 'faiss_index.index'
faiss_index = load_faiss_index_from_s3(s3_index_bucket, s3_index_key)

retrieved_chunks = retrieve_chunks_from_db(query_embedding, faiss_index)

Distances: [[0.8726399 0.880074  0.8870468 1.0624946 1.0845742]]
Indices: [[849 844 845 847 848]]
Table item: {'Items': [], 'Count': 0, 'ScannedCount': 0, 'ResponseMetadata': {'RequestId': 'ML4P6C5HRCCO6MFGL28QF6AV33VV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 06 Oct 2024 03:12:12 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '39', 'connection': 'keep-alive', 'x-amzn-requestid': 'ML4P6C5HRCCO6MFGL28QF6AV33VV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '3413411624'}, 'RetryAttempts': 0}}
No item found for FAISS index: 849
Table item: {'Items': [], 'Count': 0, 'ScannedCount': 0, 'ResponseMetadata': {'RequestId': 'TT6ESJH5PEGOBBRAE7253GA3LRVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 06 Oct 2024 03:12:12 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '39', 'connection': 'keep-alive', 'x-amzn-requestid': 'TT6ESJH5PEGOBBRAE7253GA3LRVV4

In [ ]:
retrieved_chunks

### 6. Process all files in S3 bucket

#### Create S3 bucket and upload documents to test the pipeline

In [44]:
# Define the variables
script_path = "../scripts/create_s3_bucket.sh"
bucket_name = "test-docs-loka"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x {script_path}

!source {script_path} {bucket_name} {region}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
    "Location": "http://test-docs-loka.s3.amazonaws.com/"
}
S3 bucket test-docs-loka created in region eu-north-1.


In [45]:
# Define the variables
script_path = "../scripts/upload_to_s3.py"
source_folder = "../test_data/"
bucket_name = "test-docs-loka"
folder_name = "sagemaker_documentation"

# Run the python script with the variables
!python {script_path} {source_folder} {bucket_name} {folder_name}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


File ../test_data/automating-sagemaker-with-eventbridge.md uploaded successfully to test-docs-loka/sagemaker_documentation/automating-sagemaker-with-eventbridge.md.
File ../test_data/asff-resourcedetails-awssagemaker.md uploaded successfully to test-docs-loka/sagemaker_documentation/asff-resourcedetails-awssagemaker.md.
File ../test_data/aws-properties-events-rule-sagemakerpipelineparameter.md uploaded successfully to test-docs-loka/sagemaker_documentation/aws-properties-events-rule-sagemakerpipelineparameter.md.
File ../test_data/amazon-sagemaker-toolkits.md uploaded successfully to test-docs-loka/sagemaker_documentation/amazon-sagemaker-toolkits.md.


#### Delete and create the DynamoDB table again

In [68]:
# Define the variables
table_name = "test_docs_chunks"
region = "eu-north-1"

# Navigate one directory up and make the script executable (optional)
!chmod +x ../scripts/create_dynamo_table.sh

# Run the script from one level above the current working directory
!source ../scripts/create_dynamo_table.sh {table_name} {region}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{
    "TableDescription": {
        "TableName": "test_docs_chunks",
        "TableStatus": "DELETING",
        "ProvisionedThroughput": {
            "NumberOfDecreasesToday": 0,
            "ReadCapacityUnits": 0,
            "WriteCapacityUnits": 0
        },
        "TableSizeBytes": 0,
        "ItemCount": 0,
        "TableArn": "arn:aws:dynamodb:eu-north-1:585008050272:table/test_docs_chunks",
        "TableId": "5e702efe-3ba9-4b9a-b59f-f7d53d790295",
        "BillingModeSummary": {
            "BillingMode": "PAY_PER_REQUEST",
            "LastUpdateToPayPerRequestDateTime": 1728221966.1
        },
        "DeletionProtectionEnabled": false
    }
}
DynamoDB table test_docs_chunks deleted.
{
    "TableDescription": {
        "AttributeDefinitions": [
            {
                "AttributeName": "faiss_index",
                "AttributeType": "N"
            },
            {
                "AttributeName": "id",
                "AttributeType": "S"
            }
        ],
    

### Define variables

In [69]:
# Define the variables
bucket_name = "aws-docs-loka-bk"
s3_index_bucket = 'test-docs-loka'
s3_index_key = 'faiss/faiss_index.index'

table_name = "test_docs_chunks"
region = "eu-north-1"

# Initialize the DynamoDB resource and table
dynamodb = boto3.resource('dynamodb', region_name=region)
table = dynamodb.Table(table_name)

In [70]:
def process_files(bucket_name, folder_name, s3_index_bucket, s3_index_key):
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=folder_name)
    
    all_embeddings = []
    faiss_index = 0  # Initialize the FAISS index counter

    for obj in response.get('Contents', []):
        file_key = obj['Key']  # Get the file path in S3
        print(f"Processing file: {file_key}")

        # Step 1: Download file from S3
        text = download_file_from_s3(s3, bucket_name, file_key)
        
        # Step 2: Clean and chunk the text
        chunks = chunk_text(text)
        print(f"File: {file_key} - Number of chunks: {len(chunks)}")
        
        # Step 3: Generate embeddings
        embeddings = generate_embeddings(chunks)
        
        # Step 4: Save chunks and embeddings to DynamoDB and keep track of FAISS indices
        faiss_index = save_chunks_and_embeddings_to_db(file_key, chunks, embeddings, faiss_index)
        print(f"FAISS index: {faiss_index}")
        all_embeddings.append(embeddings)

    # Combine all embeddings and create FAISS index
    if all_embeddings:
        combined_embeddings = np.vstack(all_embeddings)
        print(f"Combined embeddings shape: {combined_embeddings.shape}")
        faiss_index_obj = create_faiss_index(combined_embeddings)
        save_faiss_index_to_s3(faiss_index_obj, s3_index_bucket, s3_index_key)
    else:
        print("No valid embeddings found.")


In [71]:
folder_name = 'sagemaker_documentation/'

process_files(bucket_name, folder_name, s3_index_bucket, s3_index_key)

Processing file: sagemaker_documentation/amazon-sagemaker-toolkits.md
Downloaded sagemaker_documentation/amazon-sagemaker-toolkits.md from aws-docs-loka-bk to /tmp/downloaded_document.txt
File: sagemaker_documentation/amazon-sagemaker-toolkits.md - Number of chunks: 4
DB FAISS index: 1
DB FAISS index: 2
DB FAISS index: 3
DB FAISS index: 4
Saved chunks and embeddings for sagemaker_documentation/amazon-sagemaker-toolkits.md to DynamoDB
FAISS index: 4
Processing file: sagemaker_documentation/asff-resourcedetails-awssagemaker.md
Downloaded sagemaker_documentation/asff-resourcedetails-awssagemaker.md from aws-docs-loka-bk to /tmp/downloaded_document.txt
File: sagemaker_documentation/asff-resourcedetails-awssagemaker.md - Number of chunks: 1
DB FAISS index: 5
Saved chunks and embeddings for sagemaker_documentation/asff-resourcedetails-awssagemaker.md to DynamoDB
FAISS index: 5
Processing file: sagemaker_documentation/automating-sagemaker-with-eventbridge.md
Downloaded sagemaker_documentation

In [57]:
query = "What is SageMaker?"
query_embedding = model.encode([query])

# Load FAISS index from S3
faiss_index = load_faiss_index_from_s3(s3_index_bucket, s3_index_key)

retrieved_chunks = retrieve_chunks_from_db(query_embedding, faiss_index)

Distances: [[0.6441387  0.71701384 0.77806723 0.79565495 0.8109966 ]]
Indices: [[862 910 856 696 838]]
Table item: {'Items': [{'faiss_index': Decimal('862'), 'embedding': Binary(b'\x80\x04\x95\x8c\x06\x00\x00\x00\x00\x00\x00\x8c\x15numpy.core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01M\x80\x01\x85\x94h\x03\x8c\x05dtype\x94\x93\x94\x8c\x02f4\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89B\x00\x06\x00\x00\xafn\xc8\xbd\x143c\xbd\xddF\xea\xbd<!w=.\xa2I<\xe3h\xa1\xbd\xb9\x07\xaf\xbd2h(=\x0b\x98\xaf\xbd#1\xa5\xb9!\xfd\n=\xe0\xc2\x86:\xa0\x81T=\xa7\xb9G\xbb\xa8/\x87=\xac\x12T=;\x0c\x88\xbdV\x89\x93;N\x11Z<\xa5kd\xbd\x1apy\xbb\x99\xa39:\x87W\x8d;{:N=\xe3\x1d\x94=P\xa8\x95\xbd\xfc\xca\xae<\xfa\xf2\x07=\xf6j\x11\xbc\xf0po<\xc2\xcb\x98=&z\x1f=\xf6\x7fN=\xf3\xc0\xd4<|\xb6\xe6<\xb0\x8a\x94=\xed\x831\xbd\x95i)<\xf7h\xe8:\x8bh;\xbde\x82!\xbb~@\xd4\xbc-m\xa7\xbd\xc

In [51]:
retrieved_chunks

(['must contain a serving stack to process these requests\\. In a hosting or batch transform container, the model files are located in the same folder to which they were written during training\\. ``` /opt/ml/model │ └── <model files> ``` For more information, see [Use your own inference code](your-algorithms-inference-main.md)\\. ## Single Versus Multiple Containers<a name="sagemaker-toolkits-separate-images"></a> You can either provide separate Docker images for the training algorithm and inference code or you can use a single Docker image for both\\. When creating Docker images for use with SageMaker, consider the following: + Providing two Docker images can increase storage requirements and cost because common libraries might be duplicated\\. + In general, smaller containers start faster for both training and hosting\\. Models train faster and the hosting service can react to increases in traffic by automatically scaling more quickly\\. + You might be able to write an inference con

In [117]:
# Generate Answer using LLM
def generate_answer(query, retrieved_chunks):
    """Generate an answer using an LLM based on the query and retrieved chunks."""
    context = "\n".join(retrieved_chunks)  # Combine all chunks into a single context
    
    # Create the prompt for the LLM
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Send the prompt to the LLM (using OpenAI's GPT)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": 'system', "content": 'You are a helpful assistant.'}, 
                  {"role": 'user', "content": prompt}],
    )
    
    print(f"Response from the LLM:\n{response}")
    # Extract the answer from the response
    answer = response.choices[0].message.content
    
    return answer

In [118]:
# Generate answer
answer = generate_answer(query, retrieved_chunks[0])
print("Answer:", answer)
print("For further reading:", retrieved_chunks[1])

Response from the LLM:
ChatCompletion(id='chatcmpl-ACwXxAPbnQ6KK2JL5lkp3usjFzL8w', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='You should use MLOps because it enhances the efficiency and effectiveness of your machine learning projects. By integrating MLOps practices, you can:\n\n1. **Improve Delivery Time**: Streamlining the machine learning development lifecycle helps accelerate the deployment of ML models.\n2. **Reduce Defects**: Implementing quality assurance measures ensures higher quality models with fewer errors.\n3. **Increase Productivity**: Offering self-service environments with access to curated datasets enables data engineers and scientists to work more efficiently and minimizes time lost due to data issues.\n4. **Ensure Repeatability**: Automating the steps in the machine learning development cycle promotes a consistent and repeatable process for model training, evaluation, versioning, and deployment.\n5. **Enhance R

In [61]:
# Check how the chunks are stored
response = table.scan()

print(f"Found {response['Count']} items")

all_embeddings = []

# Iterate through the items and print chunks and embeddings
for item in response['Items']:
    # print(f"Item: {item}")
    chunk = item['chunk']
    file_key = item['file_key']
    faiss_index = item['faiss_index']
    # print(f"Chunk: {chunk}")
    print(f"File key: {file_key}")
    print(f"faiss_index: {faiss_index}")

Found 283 items
File key: sagemaker_documentation/kubernetes-sagemaker-operators-migrate.md
faiss_index: 654
File key: sagemaker_documentation/aws-properties-sagemaker-modelcard-metricgroup.md
faiss_index: 188
File key: sagemaker_documentation/kubernetes-sagemaker-components-install.md
faiss_index: 562
File key: sagemaker_documentation/train-evaluate-models-using-sagemaker-notebook.md
faiss_index: 895
File key: sagemaker_documentation/aws-properties-sagemaker-modelcard-intendeduses.md
faiss_index: 185
File key: sagemaker_documentation/neo-job-compilation-sagemaker-sdk.md
faiss_index: 680
File key: sagemaker_documentation/aws-properties-sagemaker-endpoint-trafficroutingconfig.md
faiss_index: 89
File key: sagemaker_documentation/kubernetes-sagemaker-components-install.md
faiss_index: 553
File key: sagemaker_documentation/sagemaker-geospatial-roles.md
faiss_index: 715
File key: sagemaker_documentation/common-info-all-sagemaker-models-logs.md
faiss_index: 507
File key: sagemaker_documentat